# Advanced methods for RAG applications

In this final notebook, we complete our Retrieval-Augmented Generation (RAG) pipeline with a fully modular and production-ready setup. We begin by serving three models via vLLM: a language model, an embedding model, and a re-ranker. To integrate the re-ranker, we implement a custom Haystack component that communicates with the vLLM server and processes its output for document re-ranking.

Instead of Haystack’s Document Cleaner, we now use Docling, a powerful document processing toolkit designed for GenAI workflows.

Docling supports advanced parsing of diverse formats (PDF, DOCX, HTML, etc.), and provides rich metadata extraction, layout understanding, and seamless integration with frameworks like Haystack. 

For chunking, we adopt Hybrid Chunking, which combines document structure awareness with token-based splitting. This ensures semantically coherent chunks that respect token limits—ideal for LLMs with context window constraints.

We convert and index documents using the DoclingConverter, inspect the resulting chunks and metadata, and store the embeddings in our Milvus vector database. 

The RAG pipeline is then initialized with several new enhancements:

- A query rephrasing LLM improves clarity and intent before retrieval.
- A re-ranker refines the retrieved documents for relevance.
- Conversation history is incorporated to provide contextual continuity across interactions.


### Import packages

In [ ]:
from pathlib import Path
import os
from typing import List
from haystack import Pipeline, component, Document
from haystack.components.writers import DocumentWriter
from haystack.components.embedders import OpenAIDocumentEmbedder, OpenAITextEmbedder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from docling_haystack.converter import DoclingConverter
from docling.chunking import HybridChunker
from haystack.components.rankers import SentenceTransformersSimilarityRanker
from haystack.components.joiners import ListJoiner
from haystack.components.builders import (
    ChatPromptBuilder,
    PromptBuilder,
)
from haystack.components.converters import OutputAdapter
from haystack.components.generators import OpenAIGenerator
from milvus_haystack import MilvusDocumentStore, MilvusEmbeddingRetriever
from haystack_experimental.components.retrievers import ChatMessageRetriever
from haystack_experimental.components.writers import ChatMessageWriter
from haystack_experimental.chat_message_stores.in_memory import InMemoryChatMessageStore
import textwrap
from dotenv import dotenv_values
from haystack.utils import Secret
import requests

#### Helper Functions

In [ ]:
def format_output(llm_reponse:dict)-> str: 
    formatted_text = llm_reponse["llm"]["replies"][0].text
    wrapped_text = "\n".join(
        textwrap.fill(line, width=120, subsequent_indent="  ") if line.strip() else line
        for line in formatted_text.splitlines()
    )
    return wrapped_text


# Custom Haystack Components to serve a re-ranking model with vLLM

@component
class VLLMRerankComponent:
    def __init__(self, api_url: str, api_key: str, model_name: str, top_k: int = 10, query_prefix: str = ""):
        """
        Custom Haystack component to rerank documents using vLLM's /v1/rerank endpoint.

        :param api_url: Base URL of the vLLM server (e.g., http://localhost:8000/v1)
        :param api_key: API key for authentication
        :param model_name: Name of the reranker model served by vLLM
        :param top_k: Number of top documents to return
        """
        self.api_url = api_url.rstrip("/") + "/rerank"
        self.api_key = api_key
        self.model_name = model_name
        self.top_k = top_k
        self.query_prefix = query_prefix

    @component.output_types(documents=list[Document])
    def run(self, query: str, documents: list[Document]) -> dict:
        if not documents:
            return {"documents": []}

        full_query = self.query_prefix + query

        payload = {
            "model": self.model_name,
            "query": full_query,
            "documents": [doc.content for doc in documents],
            "top_n": self.top_k
        }

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(self.api_url, json=payload, headers=headers)
        response.raise_for_status()
        result = response.json()

        # Map scores back to documents
        ranked_docs = []
        for item in result.get("results", []):
            index = item["index"]
            score = item["relevance_score"]
            doc = Document(content=documents[index].content, meta=documents[index].meta)
            doc.score = score
            ranked_docs.append(doc)

        return {"documents": ranked_docs}


#### Setup VLLM Connection Parameters

In [ ]:
llm_config = dotenv_values('/nvme/scratch/edu29/llm.env')
embd_config = dotenv_values('/nvme/scratch/edu29/embd.env')
ranker_config = dotenv_values('/nvme/scratch/edu29/ranker.env')

### Extract chunks and meta-data using Docling

In [ ]:
DOCUMENTS_DIR = Path("./dummy_data/documents_dir")
FILES = [file.resolve() for file in DOCUMENTS_DIR.rglob("*") if file.is_file()]

Using the **DoclingConverter** with a **HybridChunker** that utilizes the `"sentence‑transformers/all‑MiniLM‑L6‑v2"` tokenizer to split and process a PDF (from arXiv ID 2408.09869) into semantically coherent chunks.
 
It applies layout-aware and tokenization-aligned chunking, preparing the document for embedding with sentence-transformer models. Perfect for downstream tasks like RAG pipelines, semantic search, or document understanding.

In [ ]:
EMBED_MODEL_ID = embd_config['MODEL_NAME']

In [ ]:
converter = DoclingConverter(chunker=HybridChunker(tokenizer=EMBED_MODEL_ID))
documents = converter.run(paths=["https://arxiv.org/pdf/2408.09869"])

#### Inspect a chunk

In [ ]:
CHUNK_ID = 2
print(f'Chunk Heading: {documents["documents"][CHUNK_ID].meta["dl_meta"]["meta"]["headings"]}\n')
print("Chunk content:")
print(f'{documents["documents"][CHUNK_ID].content}\n')
doc_items = documents["documents"][CHUNK_ID].meta["dl_meta"]["meta"]["doc_items"][0]
print("Document Items:")
for k, v in doc_items.items():
    if k == 'prov':
      print(v[0])
    else:
      print(f"{k}: {v}")

### Initialize the Vector Database

Set up the Milvus vector database to store document embeddings. The `drop_old` parameter ensures any existing data is cleared.

In [ ]:
HOME_DIR = os.path.expanduser("~")
db_path = os.path.join(HOME_DIR, "rag-training", "rag_vectordb.db")

In [ ]:
connection_args={"uri": db_path}
document_store = MilvusDocumentStore(
    connection_args=connection_args,
    drop_old=True,
)

## Indexing Documents and performing RAG

Before we were using 5 components:

1. Turn them into compatible Haystack *Documents*.
2. Clean each Document using Haystack's `DocumentCleaner`. This removes any whitespaces, empty lines, specified substrings, regexes and so on.
3. Then we split our documents into *smaller chunks*. We can define various split methods and length.
4. Turn them into embeddings with an *embedder*.
5. Store them in a Haystack *Document Store* so they can be accessed later on.

---

This time we are using 3:

1. DoclingConverter
2. Embedder
3. Writer

This is because Docling will perform the following for us:

1. Document conversion
2. Document Chunking
3. Document tokenization

In [ ]:
doc_embedder = OpenAIDocumentEmbedder(api_key=Secret.from_token(embd_config['VLLM_API_KEY']),
                                     api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
                                     model=embd_config['MODEL_NAME'],)

In [ ]:
# Initialize the indexing pipeline
indexing_pipeline = Pipeline()

# Add each component to the pipeline
indexing_pipeline.add_component(
                    "converter",
                    DoclingConverter(chunker=HybridChunker(tokenizer=EMBED_MODEL_ID, max_tokens=256))
                )
indexing_pipeline.add_component("embedder", doc_embedder)
indexing_pipeline.add_component("writer", DocumentWriter(document_store))

#  Connect each component
indexing_pipeline.connect("converter", "embedder")
indexing_pipeline.connect("embedder", "writer")

# Run the Pipeline
indexing_pipeline.run({"converter": {"paths": FILES}})

#### Basic arguments for our LLM

#### Read the two templates

1. One for our RAG
2. One for our prompt-rephraser

In [ ]:
with open("./dummy_data/RAG_TEMPLATE.txt") as rag_file:
    rag_template = rag_file.read()

with open("./dummy_data/QUERY_REPHRASE_TEMPLATE.txt") as query_rephrase_file:
    query_rephrase_template = query_rephrase_file.read()
    
user_prompt_template = ChatMessage.from_user(rag_template)

print(query_rephrase_template)

#### RAG Components

For the actual RAG pipeline we have the following components:

* The *query\_rephrase\_prompt\_builder* creates a rephrasing prompt by incorporating relevant chat history.
* The *query\_rephrase\_llm* takes that prompt and generates a clearer or more contextualized version of the user’s query using an LLM.
* The *list\_to\_str\_adapter* ensures that the output of the rephrasing step is converted into a plain string for further processing.
* The *text\_embedder* then turns the rephrased query into embeddings.
* The *retriever* fetches relevant documents from the vector database using these embeddings.
* The *ranker* scores and ranks the retrieved documents based on their relevance to the query.
* The *prompt\_builder* constructs the final prompt for the LLM by combining the query, the top documents, and the conversation memory.
* The *llm* (chat\_generator) is the language model that generates the answer based on the constructed prompt.
* The *memory\_retriever* pulls past messages from the conversation to provide context and continuity.
* The *memory\_writer* stores the current interaction back into memory so it can be used in future queries.
* The *memory\_joiner* merges the latest response into the conversation history to be saved and reused.

This setup supports conversational memory, contextual query rephrasing, and hybrid RAG generation using a vector database and document reranking.


In [ ]:
query_rephrase_llm = OpenAIGenerator(api_key=Secret.from_token(llm_config['VLLM_API_KEY']),
                                     api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
                                     model=llm_config['MODEL_NAME'],)

chat_generator = OpenAIChatGenerator(api_key=Secret.from_token(llm_config['VLLM_API_KEY']),
                                     api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
                                     model=llm_config['MODEL_NAME'],)

text_embedder = OpenAITextEmbedder(api_key=Secret.from_token(embd_config['VLLM_API_KEY']),
                                     api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
                                     model=embd_config['MODEL_NAME'],)

In [ ]:
ranker_query_prefix = "Given the following user query, determine if the information provided below addresses the inquiry accurately."
ranker = VLLMRerankComponent(
    api_url=ranker_config["VLLM_LLM_URL"] + "/v1",
    api_key=ranker_config["VLLM_API_KEY"],
    model_name=ranker_config["MODEL_NAME"],
    top_k=5
)

rag_pipeline = Pipeline()
memory_store = InMemoryChatMessageStore()

# components for query rephrasing
rag_pipeline.add_component("query_rephrase_prompt_builder", PromptBuilder(query_rephrase_template))
rag_pipeline.add_component("query_rephrase_llm", query_rephrase_llm)
rag_pipeline.add_component("list_to_str_adapter", OutputAdapter(template="{{ replies[0] }}", output_type=str))
rag_pipeline.add_component("text_embedder", text_embedder)
# components for RAG
rag_pipeline.add_component("retriever", MilvusEmbeddingRetriever(document_store=document_store, top_k=15))
rag_pipeline.add_component(instance=ranker, name="ranker")
rag_pipeline.add_component("prompt_builder", ChatPromptBuilder(variables=["query", "documents", "memories"], required_variables=["query", "documents", "memories"]))
rag_pipeline.add_component("llm", chat_generator)

# components for memory
rag_pipeline.add_component("memory_retriever", ChatMessageRetriever(memory_store))
rag_pipeline.add_component("memory_writer", ChatMessageWriter(memory_store))
rag_pipeline.add_component("memory_joiner", ListJoiner(List[ChatMessage]))


# connections for query rephrasing
rag_pipeline.connect("memory_retriever", "query_rephrase_prompt_builder.memories")
rag_pipeline.connect("query_rephrase_prompt_builder.prompt", "query_rephrase_llm")
rag_pipeline.connect("query_rephrase_llm.replies", "list_to_str_adapter.replies")

rag_pipeline.connect("list_to_str_adapter.output", "prompt_builder.query")
rag_pipeline.connect("list_to_str_adapter.output", "text_embedder.text")

rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")

# connections for RAG
rag_pipeline.connect("list_to_str_adapter.output", "ranker.query") # Take the rephrased query and input it into the re-ranker
rag_pipeline.connect("retriever.documents", "ranker.documents") # Retrieve some documents and input them into the re-ranker as well
rag_pipeline.connect("ranker.documents", "prompt_builder.documents") # The re-ranker will rank the documents and return the top_k most similar ones
rag_pipeline.connect("prompt_builder.prompt", "llm.messages")
rag_pipeline.connect("llm.replies", "memory_joiner")

# connections for memory
rag_pipeline.connect("memory_joiner", "memory_writer")
rag_pipeline.connect("memory_retriever", "prompt_builder.memories")

#### Perform RAG on our data

Feel free to change the question to something else.

Our documents contain information about the following topics:

- Annual Hackathon the company is organizing
- Cybersecurity Awareness Month
- Employee Recognition Program
- New Office Layout Plan
- Office layout redesign plan
- Product X Launch Timeline
- Product Y Launch Timeline
- QuantumStream product CLI Usage
- QuantumStream product Data Encryption feature
- QuantumStream product Plugin System
- QuantumStream product REST API documentation
- QuantumStream product Scheduler feature
- QuantumStream product Scheduling tasks

---

Feel free to ask anything relating to these topics.

**Suggested prompts:**

- "Whats the purpose of the new office layout? Are we loosing our desks??"
- "I am a new employee at the company. Onboard me about the QuantumStream product."
- "I cannot find the relevant email about the Hackathon, can you tell me more details about it?"

**Note:** You can also test the conversational capabilities of the system by asking vague questions that have been covered before!

In [ ]:
if memory_store.count_messages() >= 4:
    memory_store.delete_messages()

print(memory_store.count_messages())

question = "I am a new employee at the company. Onboard me about the QuantumStream product." # Feel free to change this question

result = rag_pipeline.run(
    data={
        "query_rephrase_prompt_builder": {"query": question},
        "prompt_builder": {"template": [user_prompt_template]},
        "memory_joiner": {"values": [ChatMessage.from_user(question)]}
    },
    include_outputs_from=["llm", "query_rephrase_llm", "ranker"]
)

search_query = result['query_rephrase_llm']['replies'][0]
print(f"Search Query: {search_query}")
print(10*"#")

formatted_text = format_output(llm_reponse=result)

print(formatted_text)

#### Review the re-ranked document scores

In [ ]:
DOCUMENT_IDX = 0 # Feel free to change this number

print(result['ranker']['documents'][DOCUMENT_IDX].score)
print(result['ranker']['documents'][DOCUMENT_IDX].content)